In [1]:
import os
import shutil
from pathlib import Path

CLASS_MAPPING = {
    "bad1": "scratch",
    "bad2": "excess_material",
    "bad3": "burr",
    "bad4": "circular_cavity",
    "good": "good"
}

def get_class_from_filename(filename: str) -> str:
    """Extract class name from filename prefix."""
    for prefix, class_name in CLASS_MAPPING.items():
        if filename.startswith(prefix):
            return class_name
    return None

def prepare_classification_data(src_dir: str, dst_dir: str):
    """Organize images into classification folder structure."""
    src_path = Path(src_dir)
    dst_path = Path(dst_dir)
    
    for class_name in CLASS_MAPPING.values():
        (dst_path / class_name).mkdir(parents=True, exist_ok=True)
    
    count = 0
    for img_file in src_path.iterdir():
        if img_file.is_file() and img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            class_name = get_class_from_filename(img_file.stem)
            if class_name:
                shutil.copy2(img_file, dst_path / class_name / img_file.name)
                count += 1
    
    return count

print("Preparing training data...")
train_count = prepare_classification_data(
    "TP26_detection_v2-1/train/images", 
    "datasets/TP26/train"
)
print(f"Copied {train_count} training images")

print("Preparing validation data...")
val_count = prepare_classification_data(
    "TP26_detection_v2-1/valid/images", 
    "datasets/TP26/val"
)
print(f"Copied {val_count} validation images")

Preparing training data...
Copied 420 training images
Preparing validation data...
Copied 45 validation images


In [2]:
# Train YOLO11

from ultralytics import YOLO

# Load pre-trained classification model
model = YOLO("yolo11n-cls.pt", task="classify")

# Train the model
results = model.train(
    data="datasets/TP26",
    epochs=10,
    imgsz=224,
    project="classifyTP",
    name="train",
    exist_ok=True,
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.31 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.13.7 torch-2.10.0+cpu CPU (AMD Ryzen 7 5700U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/TP26, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, 

In [25]:
from ultralytics import YOLO
from pathlib import Path
import random

model = YOLO("C:/Users/matej/OneDrive/Dokumenty/STU FEI/Ing/TP/runs/classify/classifyTP/train/weights/best.pt")

val_dir = Path("TP26_detection_v2-1/valid/images")
sample_images = list(val_dir.glob("*.jpg"))
sample_image = sample_images[random.randint(0, len(sample_images) - 1)] if sample_images else None

if sample_image:
    print(f"Testing with: {sample_image.name}")
    results = model.predict(str(sample_image))
    result = results[0]
    pred_class = result.names[result.probs.top1]
    pred_conf = result.probs.top1conf.item()
    print(f"\nPredicted class: {pred_class}")
    print(f"Confidence: {pred_conf:.4f}")
    print(f"\nAll probabilities:")
    for name, prob in zip(result.names.values(), result.probs.data):
        print(f"  {name}: {prob:.4f}")
else:
    print("No validation images found!")

Testing with: excess_material_RPi_CM3_38_original_png.rf.a16ec55beb6295d70931a5dd34a57fa7.jpg

image 1/1 c:\Users\matej\OneDrive\Dokumenty\STU FEI\Ing\TP\YOLO\TP26_detection_v2-1\valid\images\excess_material_RPi_CM3_38_original_png.rf.a16ec55beb6295d70931a5dd34a57fa7.jpg: 224x224 excess_material 0.75, circular_cavity 0.19, burr 0.05, good 0.00, scratch 0.00, 35.5ms
Speed: 8.4ms preprocess, 35.5ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)

Predicted class: excess_material
Confidence: 0.7544

All probabilities:
  burr: 0.0496
  circular_cavity: 0.1919
  excess_material: 0.7544
  good: 0.0037
  scratch: 0.0003


In [19]:
# Calculate Validation Accuracy

from ultralytics import YOLO
from pathlib import Path

model = YOLO("C:/Users/matej/OneDrive/Dokumenty/STU FEI/Ing/TP/runs/classify/classifyTP/train/weights/best.pt")

val_dir = Path("TP26_detection_v2-1/valid/images") 

# Class mapping for ground truth
CLASS_MAPPING = {
    "bad1": "scratch",
    "bad2": "excess_material",
    "bad3": "burr",
    "bad4": "circular_cavity",
    "good": "good"
}

def get_true_class(filename: str) -> str:
    """Get true class from filename prefix."""
    for prefix, class_name in CLASS_MAPPING.items():
        if filename.startswith(prefix):
            return class_name
    return None

# Evaluate on all validation images
correct = 0
total = 0

for img_path in val_dir.glob("*.jpg"):
    true_class = get_true_class(img_path.stem)
    if true_class is None:
        continue
    
    results = model.predict(str(img_path), verbose=False)
    pred_class = results[0].names[results[0].probs.top1]
    
    if pred_class == true_class:
        correct += 1
    
    total += 1

accuracy = correct / total * 100 if total > 0 else 0
print(f"Validation Results:")
print(f"  Correct: {correct}/{total}")
print(f"  Accuracy: {accuracy:.2f}%")

Validation Results:
  Correct: 45/45
  Accuracy: 100.00%
